In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### AMPDeep dataset curation

This notebook builds a curated dataset from **AMPDeep** by integrating multiple raw inputs distributed across different formats and folder structures (FASTA files and several CSV subsets). The pipeline standardizes sequence fields, infers/assigns labels when possible, separates **modified** peptides from **non-modified** peptides, performs duplicate consistency checks, and exports the final datasets together with metadata.

- **Toxic effect / endpoint:** hemolytic
- **Source:** AMPDeep
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads raw inputs from multiple AMPDeep sources**, including:
  - FASTA/FA/TXT files from the AMPDeep input directory,
  - CSV files under `combined/`, `hemolytic/`, and `hlppredfuse/`,
  - `rnnamp` subsets (full, train, test),
  - DAASP_RNN and peptides_complete tables.
- **Standardizes sequences** by removing whitespace and mapping columns to a unified schema:
  - `sequence` (string)
  - `label` (0/1 when available)
- **Infers labels for FASTA records** from filename conventions (e.g., `class0/negative` → 0, `class1/positive` → 1). Files without label hints keep `label = NA`.
- **Separates modified peptides** into a dedicated table using two strategies:
  - explicit N/C-terminus modification columns (e.g., `N terminus`, `C terminus`),
  - text-based modification markers (e.g., tokens like `AMD` or `ACT`, or mismatches involving `original_sequence`).
- **Runs duplicate checks** by sequence:
  - keeps unique sequences,
  - collapses duplicates when labels are consistent,
  - flags sequences with conflicting labels as errors (exported for manual review).
- **Builds metadata** from the project-wide Excel description sheet and appends QC counters.
- **Exports curated datasets** (non-modified + modified) and error lists as CSV, plus a JSON metadata file.

In [2]:
name_source = "AMPDeep"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
input_dir_fasta = Path(PATH_INPUT) / name_source
valid_ext = {".fasta", ".fa", ".faa", ".txt"}
dfs = []

for file in input_dir_fasta.iterdir():
    if file.is_file() and file.suffix.lower() in valid_ext:
        df = read_fasta_doc(file)
        df["source_file"] = file.name
        dfs.append(df)

df_fasta = pd.concat(dfs, ignore_index=True)

In [4]:
folders = ["combined", "hemolytic", "hlppredfuse"]
dfs = []

for folder in folders:
    for file in (Path(PATH_INPUT) / name_source / folder).glob("*.csv"):
        df = pd.read_csv(file)
        dfs.append(df)
 
df_folders = pd.concat(dfs, ignore_index=True)

In [5]:
df_DAASP_RNN = pd.read_csv(f"{PATH_INPUT}/{name_source}/rnnamp/DAASP_RNN_dataset.csv")

In [6]:
df_peptide_complete = pd.read_csv(f"{PATH_INPUT}/{name_source}/rnnamp/peptides_complete.csv")

/tmp/ipykernel_47669/1558895454.py:1: DtypeWarning: Columns (0: HEMOLITIC CYTOTOXIC ACTIVITY - IONIC STRENGTH, 1: HEMOLITIC CYTOTOXIC ACTIVITY - SALT TYPE) have mixed types. Specify dtype option on import or set low_memory=False.
  df_peptide_complete = pd.read_csv(f"{PATH_INPUT}/{name_source}/rnnamp/peptides_complete.csv")


In [7]:
df_rnnamp = pd.read_csv(f"{PATH_INPUT}/{name_source}/rnnamp/rnnamp.csv")

In [8]:
df_rnnamp_test = pd.read_csv(f"{PATH_INPUT}/{name_source}/rnnamp/rnnamp_test.csv")

In [9]:
df_rnnamp_train = pd.read_csv(f"{PATH_INPUT}/{name_source}/rnnamp/rnnamp_train.csv")

- Concatenating dataset

In [10]:
df_fasta["label"] = pd.NA  # valor por defecto

mask_neg = df_fasta["source_file"].str.contains(
    r"class0|negative",
    case=False,
    regex=True,
    na=False
)

mask_pos = df_fasta["source_file"].str.contains(
    r"class1|positive",
    case=False,
    regex=True,
    na=False
)

df_fasta.loc[mask_neg, "label"] = 0
df_fasta.loc[mask_pos, "label"] = 1

df_fasta = df_fasta[["sequence", "label"]]

In [11]:
mask_no_mod = df_DAASP_RNN["N terminus"].isna() & df_DAASP_RNN["C terminus"].isna()

df_non_modified_DAASP_RNN = (
    df_DAASP_RNN.loc[mask_no_mod, ["Sequence", "activity"]]
    .rename(columns={
        "Sequence": "sequence",
        "activity": "label"
    })
    .reset_index(drop=True)
)

In [12]:
df_modified_DAASP_RNN = (
    df_DAASP_RNN.loc[~mask_no_mod, ["Sequence", "activity", "N terminus", "C terminus"]]
    .rename(columns={
        "Sequence": "sequence",
        "activity": "label"
    })
    .reset_index(drop=True)
)

In [13]:
mask_no_mod = df_peptide_complete["N TERMINUS"].isna() & df_peptide_complete["C TERMINUS"].isna()

df_non_modified_peptide_complete = (
    df_peptide_complete.loc[mask_no_mod, ["SEQUENCE"]]
    .rename(columns={"SEQUENCE": "sequence"})
    .assign(label=1)
    .reset_index(drop=True)
)

In [14]:
df_modified_peptide_complete = (
    df_peptide_complete.loc[~mask_no_mod, ["SEQUENCE", "N TERMINUS", "C TERMINUS"]]
    .rename(columns={
        "SEQUENCE": "sequence",
    })
    .assign(label=1)
    .reset_index(drop=True)
)

In [15]:
pattern = r'^(A M D|A C T)\b|\b(A M D|A C T)$'

df_rnnamp["is_modified"] = (
    df_rnnamp["text"].str.contains(pattern, regex=True, na=False)
    |
    df_rnnamp.apply(
        lambda row: (
            pd.notna(row["original_sequence"]) and
            pd.notna(row["text"]) and
            row["original_sequence"] in row["text"]
        ),
        axis=1
    )
)

/tmp/ipykernel_47669/2607357084.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_rnnamp["text"].str.contains(pattern, regex=True, na=False)


In [16]:
df_modified_rnnamp = df_rnnamp[df_rnnamp["is_modified"]]
df_modified_rnnamp = (
    df_modified_rnnamp
    .rename(columns={"text": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)

In [17]:
df_rnnamp.loc[~df_rnnamp["is_modified"], "original_sequence"] = df_rnnamp["text"]
df_non_modified_rnnamp = (
    df_rnnamp
    .rename(columns={"original_sequence": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)

In [18]:
df_rnnamp_test = (
    df_rnnamp_test
    .rename(columns={"text": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)

In [19]:
df_rnnamp_train = (
    df_rnnamp_train
    .rename(columns={"text": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)

In [20]:
df_folders = (
    df_folders
    .rename(columns={"text": "sequence", "labels": "label"})
    .assign(
        sequence=lambda d: d["sequence"].astype(str).str.replace(" ", "", regex=False)
    )
    [["sequence", "label"]]
)

In [21]:
df_ampdeep = pd.concat(
    [df_fasta, 
    df_folders, 
    df_non_modified_DAASP_RNN,
    df_non_modified_peptide_complete, 
    df_non_modified_rnnamp,
    df_rnnamp_test,
    df_rnnamp_train],
    ignore_index=True
)
df_ampdeep.shape

(102199, 2)

In [22]:
df_modified_peptide_complete = df_modified_peptide_complete.rename(columns={
    "N TERMINUS": "N terminus",
    "C TERMINUS": "C terminus"
})

df_ampdeep_modified = pd.concat(
    [df_modified_DAASP_RNN,
     df_modified_peptide_complete,
     df_modified_rnnamp],
    ignore_index=True
)

df_ampdeep_modified.shape

(86432, 4)

- Checking duplicates

In [23]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_ampdeep, group_seq="sequence", sort_key="label")

In [24]:
df_remove_duplicated_mod, df_errors_mod, df_unique_mod = processing_duplicated(df_ampdeep_modified, group_seq="sequence", sort_key="label")

In [25]:
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [26]:
df_full_mod = pd.concat([df_unique_mod, df_remove_duplicated_mod], axis=0)


In [27]:
df_errors.shape

(1469, 1)

In [28]:
df_errors_mod.shape

(872, 1)

- Working with metada

In [29]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [30]:
raw_total_sequences = (
    len(df_fasta)
    + len(df_folders)
    + len(df_DAASP_RNN)
    + len(df_peptide_complete)
    + len(df_rnnamp)
    + len(df_rnnamp_test)
    + len(df_rnnamp_train)
)

In [31]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'MIT',
 'year of publication': 2022,
 'last update date': datetime.datetime(2022, 9, 28, 0, 0),
 'download date': Timestamp('2025-04-01 00:00:00'),
 'file format': 'csv;fasta',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative;Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Randomly generated sequences, Sampling from Swiss-Prot;No information',
 'repository or server': 'https://github.com/milad73s/AMPDeep/tree/main/data',
 'publication': 'https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-022-04952-z#Sec12',
 'number_of_raw_sequences': 186074,
 'number_of_sequences_retained': 16191,
 'number_of_positive_sequences': 9449,
 'number_of_negative_sequences': 6742,
 'number_of_erroneous_sequences': 1469,
 'modified_sequences_included': False}

- Exporting data

In [32]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [33]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_mod.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/modified_hemolytic_dataset.csv", index=False)

df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)
df_errors_mod.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_modified_sequences.csv", index=False)